# SentimentumAI — HuggingFace Inference (Google Colab)

Este notebook corre la inferencia del modelo `cardiffnlp/twitter-roberta-base-hate-latest` sobre el test set completo y exporta los resultados para integrarlos en el notebook principal.

**Pasos:**
1. Subir `twitter_train.csv` cuando se pida
2. Ejecutar todas las celdas en orden
3. Descargar `hf_resultados.csv` al finalizar

> Activar GPU: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`

In [1]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

if device == "cuda":
  print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
  print("Sin GPU")

Device: cuda
GPU: Tesla T4


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
df_twitter = pd.read_csv("Datos/twitter_train.csv")

_, X_test_text, _, y_test = train_test_split(
    df_twitter['tweet'],
    df_twitter['label'],
    test_size=0.2,
    random_state=42,
    stratify=df_twitter['label']
)

tweets_origin = X_test_text.to_list()
y_muestra = y_test.values

print(f"Test set: {len(tweets_origin)} tweets")
print(f"No ODIO: {(y_muestra==0).sum()}  | ODIO: {(y_muestra==1).sum()}")

Test set: 6393 tweets
No ODIO: 5945  | ODIO: 448


In [4]:
from transformers import pipeline as hf_pipeline

print(f"Cargando modelo en {device}...")
hf_model = hf_pipeline(
    task="text-classification",
    model= "cardiffnlp/twitter-roberta-base-hate-latest",
    device= 0 if device == "cuda" else -1,
    truncation= True,
    max_lenght= 120
)
print("Modelo cargado")

BATCH_SIZE= 64
resultados = []

for i in range(0, len(tweets_origin), BATCH_SIZE):
  batch = tweets_origin[i: i+BATCH_SIZE]
  resultados.extend(hf_model(batch))

label_map = {"HATE": 1, "NOT-HATE": 0}
y_pred_hf = np.array([label_map[r["label"]] for r in resultados])
y_prob_hf = np.array([r['score'] if r['label']=='HATE' else 1-r['score'] for r in resultados])

print(f"\n Completado | Predcciones de odio: {(y_pred_hf).sum()}")

Cargando modelo en cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo cargado


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



 Completado | Predcciones de odio: 295


In [5]:
df_export = pd.DataFrame({
    'tweet':tweets_origin,
    'label_real': y_muestra,
    'label_predict':y_pred_hf,
    'prob_hf': y_prob_hf
})

df_export.to_csv("hf_resultados.csv", index=False)
print(f"hf_resultados.csv exportado correctamente [{len(df_export)} filas]")

hf_resultados.csv exportado correctamente [6393 filas]


## Fine Tunning sobre dataset

Re-entrenamos el modelo con los tweets de entrenamiento del dataset para que aprenda la definición en particular del set de datos de hate speech del proyecto.

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| Epochs | 3 | Balance entre aprendizaje y overfitting |
| Batch size | 32 | Óptimo para T4 16GB |
| Learning rate | $2e-5$ | Estándar para fine-tuning de Transformers |
| Max length | 128 | Cubre el 95% de tweets sin truncar |
| Class weight | Balanceado | Compensa desbalance 13:1 |

In [6]:
df = pd.read_csv('Datos/twitter_train.csv')

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['tweet'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label']
)

tweets_train = X_train_text.tolist()
tweets_test  = X_test_text.tolist()
y_train_arr  = y_train.values
y_test_arr   = y_test.values

In [7]:
#!pip install datasets --quiet
"""
Fine-Tunning - cardiffnlp/twitter-roberta-base-hate-latest
"""
from transformers import (
  AutoTokenizer, AutoModelForSequenceClassification,
  TrainingArguments, Trainer
)

from torch.utils.data import Dataset
from sklearn.utils.class_weight import compute_class_weight
import torch.nn as nn

MODEL_NAME = 'cardiffnlp/twitter-roberta-base-hate-latest'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_ft = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True
)
print("Modelo base cargado para fine tuning")

from datasets import Dataset as HFDataset

def tokenize(examples):
    return tokenizer(examples['text'], truncation=True,
                     padding="max_length", max_length=128)

train_hf = HFDataset.from_dict({
    'text':   tweets_train,
    'labels': y_train_arr.tolist()
}).map(tokenize, batched=True)

test_hf = HFDataset.from_dict({
    'text':   tweets_test,
    'labels': y_test_arr.tolist()
}).map(tokenize, batched=True)

train_hf.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_hf.set_format('torch',  columns=['input_ids', 'attention_mask', 'labels'])

print(f"✅ Train: {len(train_hf)} | Test: {len(test_hf)}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo base cargado para fine tuning


Map:   0%|          | 0/25569 [00:00<?, ? examples/s]

Map:   0%|          | 0/6393 [00:00<?, ? examples/s]

✅ Train: 25569 | Test: 6393


In [8]:
"""
Fine-Tunning - cardiffnlp/twitter-roberta-base-hate-latest
"""

MODEL_NAME = 'cardiffnlp/twitter-roberta-base-hate-latest'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_ft = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True
)
print("Modelo base cargado para fine tuning")

class TweetDataset(Dataset):
  def __init__(self, texts, labels, tokenizer, max_length=128):
    self.texts = texts
    self.labels = torch.tensor(labels, dtype=torch.long)
    self.tokenizer = tokenizer
    self.max_length = max_length

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    enc = self.tokenizer(
        self.texts[idx],
        truncation = True,
        padding = "max_length",
        max_length = self.max_length,
        return_tensors = "pt"
    )
    return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         self.labels[idx]
        }

train_dataset = TweetDataset(tweets_train, y_train_arr.tolist(), tokenizer)
test_dataset  = TweetDataset(tweets_test,  y_test_arr.tolist(),  tokenizer)
print(f"Datasets | Train: {len(train_dataset)} | Test: {len(test_dataset)}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo base cargado para fine tuning
Datasets | Train: 25569 | Test: 6393


In [9]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_arr),
    y=y_train_arr
)

weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float
).to(device)

class WeightedTrainer(Trainer):
  def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
    labels = inputs.pop("labels")
    outputs = model(**inputs)
    loss = nn.CrossEntropyLoss(weight=weights_tensor)(outputs.logits, labels)
    return (loss, outputs) if return_outputs else loss


training_args = TrainingArguments(
    output_dir = "./ft_model",
    num_train_epochs = 3,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size= 64,
    learning_rate = 2e-5,
    weight_decay = 0.01,
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    load_best_model_at_end = True,
    logging_steps = 100,
    fp16 = True if device == 'cuda' else False,
    report_to = 'none'
    )

trainer = WeightedTrainer(
    model = model_ft,
    args = training_args,
    train_dataset = train_hf,
    eval_dataset = test_hf
)

print("Iniciando fine-tunnning ...")
trainer.train()
print("Fine Tunnning completado")

Iniciando fine-tunnning ...


Epoch,Training Loss,Validation Loss
1,0.239533,0.278787
2,0.145865,0.224738
3,0.041640,0.392793


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine Tunnning completado


In [10]:
import torch.nn.functional as F

model_ft.eval()
model_ft.to(device)

y_pred_ft_list = []
y_prob_ft_list = []

BATCH_SIZE = 64
for i in range (0, len(test_hf), BATCH_SIZE):
    batch = [test_dataset[j] for j in range(i, min(i+BATCH_SIZE, len(test_dataset)))]
    input_ids      = torch.stack([b['input_ids']      for b in batch]).to(device)
    attention_mask = torch.stack([b['attention_mask'] for b in batch]).to(device)

    with torch.no_grad():
        outputs = model_ft(input_ids=input_ids, attention_mask=attention_mask)
        probs   = F.softmax(outputs.logits, dim=-1)
        preds   = torch.argmax(probs, dim=-1)

    y_pred_ft_list.extend(preds.cpu().numpy())
    y_prob_ft_list.extend(probs[:, 1].cpu().numpy())
    print(f"  Procesados: {min(i+BATCH_SIZE, len(test_dataset))}/{len(test_dataset)}", end='\r')

y_pred_ft = np.array(y_pred_ft_list)
y_prob_ft = np.array(y_prob_ft_list)

print(f" Fine-tuned | Predicciones de odio: {(y_pred_ft==1).sum()}")

 Fine-tuned | Predicciones de odio: 477


In [11]:
"""
============================================================
Exportar resultados fine-tuned
============================================================
"""

from google.colab import files

pd.DataFrame({
    'tweet':         tweets_test,
    'label_real':    y_test_arr,
    'label_predict': y_pred_ft,
    'prob_hf':       y_prob_ft.round(4)
}).to_csv('hf_finetuned_resultados.csv', index=False)
print("hf_finetuned_resultados.csv exportado")
files.download('hf_finetuned_resultados.csv')

hf_finetuned_resultados.csv exportado


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>